> **Meeting 9, as we went through it in class.** This is the notebook on the
> screen, with the instructor's notes removed. Your own notebook for the meeting —
> the one with the studio — is in the same folder.

# STAT 764
## Classification foundations

### Meeting 9 — Tuesday, October 6

In [ ]:
# Setup. Run before they walk in.
import os
import pathlib
import sys

here = pathlib.Path.cwd()
cands = []
if os.environ.get("STAT764_REPO"):
    cands.append(pathlib.Path(os.environ["STAT764_REPO"]))
cands += [p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
cands += [c.parent.parent for c in here.glob("*/course/stat764.py")]
cands += [p / "stat764-fall2026" for p in [here, *here.parents]]
cfg = next((p / "course.yaml" for p in [here, *here.parents]
            if (p / "course.yaml").exists()), None)
if cfg:
    for line in cfg.read_text().splitlines():
        if line.startswith("student_repo:"):
            cands.append(pathlib.Path(line.split(":", 1)[1].strip()))

repo = next((c for c in cands if (c / "course" / "stat764.py").exists()), None)
if repo is None:
    raise RuntimeError(
        "Cannot find the course files.\n"
        f"  looked from: {here}\n"
        "  Fix: set STAT764_REPO, or run from a folder with course/stat764.py beside it.")
os.environ["STAT764_REPO"] = str(repo)
sys.path.insert(0, str(repo / "course"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw = diabetes()
if ".venv" not in sys.executable:
    print("⚠ not the course environment (the uv .venv) — numbers may differ slightly from class")
print(f"ready — {len(raw):,} encounters, {raw['patient_nbr'].nunique():,} patients")

# Thursday's studio — nested cross-validation, ridge against the lasso

## The same nested CV for ridge and the lasso. What α did each outer fold pick?

| outer fold | 1 | 2 | 3 | 4 | 5 | nested score |
|---|---|---|---|---|---|---|
| ridge | 10 | 10 | 10 | 10 | 3.16 | 0.890 |
| lasso | 100 | 1,000 | 300 | 100 | 100 | 0.875 |

### What this shows

**Ridge's search picks nearly the same α every time. The lasso's wanders across a factor
of ten.** Near the top, the lasso's CV scores are almost tied — 0.875 for α = 100 against
0.871 for α = 300 in fold 1 — so which α wins is close to a coin flip. That is what the
outer folds' picks are good for: a **stability check**. They are thrown away; the number
you report is still the nested score, and the model you use still comes from one search on
all the data.

**Watch nested CV step by step:**
[the nested cross-validation animation, on the course site](https://datascienceuwl.github.io/STAT764/nested-cv/)

**Read the whole studio, worked:** `course/reference/ref_day08_studio.ipynb` — the lasso's nested
CV run by hand, fold by fold, and which α you actually use. It is also in the Week 4 module.

# Lab 4 in tonight · Lab 5 and Mini-project 1 out today

## **Lab 4** — due tonight, 11:59 pm

## **Lab 5 — The metric picks the winner** — out today, due Tue Oct 13

## **Mini-project 1** — an honest prediction for a bank — out today, due Thu Oct 22.
## Teams of three, announced Thursday.

### `git pull` first. Restart & Run All. Upload the file you ran.

# Today: classification foundations

Since October 2012, Medicare has cut payments to hospitals whose patients come back within
30 days more often than expected — at first after heart attack, heart failure and
pneumonia, and by up to 3% of what Medicare pays the hospital for inpatient care. So
hospitals build models to decide who gets a follow-up call, a pharmacist's review, a
nurse's visit.

That model is a classifier, and a classifier is **two things, not one**: a probability for
each patient, and a rule that turns the probability into an action. A lot of what goes
wrong with classifiers goes wrong because the second thing was chosen by default.

Today, on about 100,000 real hospital encounters for patients with diabetes:

1. **Define the problem** before touching a model.
2. **A model that looks accurate** — until you ask what doing nothing scores.
3. **The probability and the rule**, pulled apart.
4. **The confusion matrix**, one threshold at a time.
5. **AUC** — what it tells you, and what it cannot.
6. **Precision depends on who you are predicting for.**
7. **The unit you predict for** — the same patient on both sides of the split. Meeting 6's
   exit ticket asked for a dataset where shuffling rows would be dishonest; three of you
   named this one. Today it is measured.

**The question for today.** When a patient with diabetes leaves the hospital, will they be
back within 30 days? A hospital that could answer that at discharge could make the call,
the review or the visit before the patient walks out the door. Until now our outcome has been
a number — a sale price. Today it is yes or no, and the first decision is what exactly
counts as *yes*.

# Before any model: what are we predicting?

## Readmitted **within 30 days** — yes or no.

## And some of these encounters cannot be readmitted at all.

In [ ]:
# What one row is: one hospital STAY (an "encounter"), not one patient. The same patient
# can appear many times -- patient_nbr is how you tell.
PREVIEW = ["patient_nbr", "age", "gender", "time_in_hospital", "num_medications",
           "number_inpatient", "A1Cresult", "insulin", "discharge_disposition_id",
           "readmitted"]
raw[PREVIEW].sample(6, random_state=5)          # six stays drawn at random

In [ ]:
ENDED = [11, 13, 14, 19, 20, 21]        # discharge codes for death or hospice
ended = raw["discharge_disposition_id"].isin(ENDED)
print(f"  encounters ending in death or hospice    {ended.sum():,}")
print(f"  readmitted within 30 days, of those      {(raw.loc[ended, 'readmitted'] == '<30').mean():.1%}")
print(f"  readmitted within 30 days, of the rest   {(raw.loc[~ended, 'readmitted'] == '<30').mean():.1%}")

d = raw[~ended].copy()
d["readmit30"] = (d["readmitted"] == "<30").astype(int)
print(f"\n  kept: {len(d):,} encounters from {d['patient_nbr'].nunique():,} patients")

In [ ]:
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
d[CAT] = d[CAT].astype(str)            # the codes are labels, not quantities


def pipeline():
    """The pipeline since Meeting 2 -- fill gaps, scale, one-hot encode, then the model."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000)),
    ])


visits = d.groupby("patient_nbr").size()
train_rows, test_rows = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=764)
                             .split(d, groups=d["patient_nbr"]))
train, test = d.iloc[train_rows], d.iloc[test_rows]
X_train, y_train = train[NUM + CAT], train["readmit30"]
X_test, y_test = test[NUM + CAT], test["readmit30"]

print(f"  patients with more than one encounter   {(visits > 1).sum():,}   (one of them {visits.max()} times)")
print(f"  train {len(train):,} encounters · test {len(test):,} encounters · "
      f"patients in both: {len(set(train['patient_nbr']) & set(test['patient_nbr']))}")

# Is this a good model?

## A logistic regression, scored on patients it has never seen.

In [ ]:
model = pipeline().fit(X_train, y_train)
print(f"  accuracy on {len(test):,} unseen encounters   {accuracy_score(y_test, model.predict(X_test)):.4f}")

# What would doing nothing score?

## Predict "not readmitted" for every single encounter.

In [ ]:
nobody = np.zeros(len(y_test), dtype=int)
caught = int(((model.predict(X_test) == 1) & (y_test.to_numpy() == 1)).sum())

print(f"  predict 'not readmitted' for everyone   accuracy {accuracy_score(y_test, nobody):.4f}")
print(f"  the model                               accuracy {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"\n  readmissions in the test set            {int(y_test.sum()):,}")
print(f"  readmissions the model flagged          {caught}")

# Less accurate than doing nothing.

## It flagged **48** of the **2,739** patients who came back.

### Nothing is broken. Accuracy rewards ignoring the rare class.

# A classifier is a probability and a rule

In [ ]:
p = model.predict_proba(X_test)[:, 1]        # the model's estimate of P(readmitted within 30 days)

print(f"  predicted probabilities: median {np.median(p):.3f}, mean {p.mean():.3f}, "
      f"largest {p.max():.3f}")
print(f"  share of encounters above 0.5        {np.mean(p > 0.5):.2%}   ({(p > 0.5).sum()} of {len(p):,})")
print(f"  model.predict is exactly 'p > 0.5'   {np.array_equal(model.predict(X_test), (p > 0.5).astype(int))}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(p, bins=60, color="grey")
ax.axvline(0.5, color="black", ls="--")
ax.annotate("  the default rule: 0.5", (0.5, ax.get_ylim()[1] * 0.8))
ax.set_xlabel("predicted probability of readmission within 30 days")
ax.set_ylabel("encounters")
plt.show()

# The rule is the problem

## 0.5 is not part of the model. scikit-learn chose it.

## It means: act only on patients **more likely than not** to come back.

# The confusion matrix

The whole map first — the four boxes, and every rate that is built from them:

![An extended confusion matrix. Rows are what actually happened: came back, or did not. Columns are what the rule did: flagged, or not flagged. The four boxes are true positive (caught), false negative (missed, a Type II error), false positive (a false alarm, a Type I error) and true negative (rightly left alone). To the right, read across a row: sensitivity equals TP over TP plus FN, and specificity equals TN over FP plus TN, with the false negative and false positive rates. Below, read down a column: precision equals TP over TP plus FP, and negative predictive value equals TN over FN plus TN. Also prevalence and accuracy, which use all four boxes.](extended_confusion_matrix.png)

## Every encounter lands in one of four boxes. At the default rule, 0.5:

In [ ]:
# A threshold turns probabilities into decisions. The confusion matrix counts the decisions.
flag = p >= 0.5                                  # the default rule: flag if p is at least 0.5

# Rows: what actually happened. Columns: what the rule did. pd.crosstab counts every pair,
# which gives the four boxes with their names on them.
actual = pd.Series(np.where(y_test == 1, "came back", "did not come back"), name="actually")
rule = pd.Series(np.where(flag, "flagged", "not flagged"), name="the rule at 0.5")
print(pd.crosstab(actual, rule).loc[["came back", "did not come back"], ["flagged", "not flagged"]])

# Two kinds of error

| | **flagged** | **not flagged** |
|---|---|---|
| **came back** | **true positive (TP)** — caught | **false negative (FN)** — missed |
| **did not come back** | **false positive (FP)** — a false alarm | **true negative (TN)** — rightly left alone |

## A false alarm is a **Type I error**. A miss is a **Type II error**. Sensitivity is **power**.

# Every rate is a ratio of the four boxes

| name | also called | from the boxes | in words |
|---|---|---|---|
| **sensitivity** | recall, true positive rate; *power* | TP / (TP + FN) | of those who **came back**, the share you flagged |
| **specificity** | true negative rate | TN / (TN + FP) | of those who **did not**, the share you left alone |
| **false positive rate** | 1 − specificity; *the Type I error rate* | FP / (FP + TN) | of those who did not, the share you flagged anyway |
| **false negative rate** | 1 − sensitivity; *the Type II error rate* | FN / (FN + TP) | of those who came back, the share you missed |
| **precision** | positive predictive value (PPV) | TP / (TP + FP) | of those you **flagged**, the share who came back |
| **negative predictive value** | NPV | TN / (TN + FN) | of those you **did not flag**, the share who did not come back |
| **accuracy** | | (TP + TN) / everyone | the share the rule got right |

For this problem, at the default rule — the four boxes above:

- **Sensitivity** — of the 2,739 patients who came back within 30 days, the rule flagged 48:
  **1.8%**. A nurse working from this list would never hear about 98 of every 100
  readmissions.
- **Specificity** — of the 21,967 who did not come back, the rule left 21,913 alone:
  **99.8%**. Almost no false alarms — because it almost never flags anyone.
- **Precision** — of the 102 patients it flagged, 48 came back: **47%**. When the rule does
  speak it is right about half the time; it just speaks for 102 of 24,706 patients.

## Sensitivity and specificity read **across a row**: they start from what happened.
## Precision and NPV read **down a column**: they start from what you did.

# Same probabilities, three rules

In [ ]:
# The model never changes below -- only the threshold does. Each row is one rule.
rows = []
for threshold in (0.5, 0.2, 0.11):
    flag = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, flag).ravel()   # the four boxes, as numbers
    rows.append({"threshold": threshold, "flagged": flag.mean(),
                 "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                 "precision": tp / (tp + fp), "accuracy": (tp + tn) / len(flag)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Every number moved. The model did not.

## Move the threshold from 0.5 to 0.11 and sensitivity climbs from 0.018 to 0.572, while
## precision falls from 0.471 to 0.168 and accuracy from 0.889 to 0.639.

# From the lab — the same four boxes, in breast cancer

My research group works with Mayo Clinic Health System on deep-learning models that read
breast ultrasound exams and score how likely a lesion is to be cancer. One of them, on 12,357
exams from 2018–2024 that it never saw during training: 1,004 cancers (8%), AUC 0.93. Saying
"benign" to everyone would be 92% accurate.

At a threshold that catches 95% of the cancers, per 1,000 women scanned (rounded):

| | **model flags** | **model clears** |
|---|---|---|
| **cancer** | 77 caught | **4 missed** — a Type II error |
| **no cancer** | **442 flagged** — a Type I error: a biopsy, a wait, a frightening week | 476 rightly cleared |

The same model at other thresholds, per 1,000 women:

| threshold catches | cancers caught | cancers missed | healthy women flagged |
|---|---|---|---|
| 99% of cancers | 80 | 1 | 776 |
| 95% | 77 | 4 | 442 |
| 90% | 73 | 8 | 254 |
| 80% | 65 | 16 | 51 |
| 70% | 57 | 24 | 14 |

The two errors are not the same mistake. A false alarm costs a biopsy; a miss can cost a
delayed diagnosis. So the clinic lives near the top of that table and accepts a great many
false alarms — and this model only advises: a radiologist still reads every exam. Where to
stop is Thursday's question.

# AUC — a ranking, not a rate

In [ ]:
auc = roc_auc_score(y_test, p)

rng = np.random.default_rng(764)
came_back = rng.choice(p[y_test.to_numpy() == 1], 100_000)
did_not = rng.choice(p[y_test.to_numpy() == 0], 100_000)

print(f"  AUC                                                          {auc:.4f}")
print(f"  100,000 random pairs (one readmitted, one not): the readmitted")
print(f"  patient got the higher probability in                        {np.mean(came_back > did_not):.4f}")

fpr, tpr, cut = roc_curve(y_test, p)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f"logistic regression, AUC {auc:.3f}")
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="a coin, AUC 0.5")
for t in (0.5, 0.2, 0.11):
    k = np.argmin(np.abs(cut - t))
    ax.plot(fpr[k], tpr[k], "o", color="black")
    ax.annotate(f"  threshold {t}", (fpr[k], tpr[k]))
ax.set_xlabel("1 − specificity (share of the not-readmitted you flagged)")
ax.set_ylabel("sensitivity (share of the readmitted you flagged)")
ax.legend(loc="lower right")
plt.show()

# 0.655: pick one who came back, one who did not

## The model ranks them correctly about **65%** of the time. A coin: **50%**.

### It cannot tell you which threshold. It does not check the probabilities.

# Precision depends on who you are predicting for

## Same model, same threshold — different base rates.

In [ ]:
flag = p >= 0.2
sens = flag[y_test.to_numpy() == 1].mean()
spec = (~flag)[y_test.to_numpy() == 0].mean()

times = d.groupby("patient_nbr")["readmit30"].transform("size").clip(upper=5)
base_rate = d.groupby(times)["readmit30"].mean()

print(f"  threshold 0.2: sensitivity {sens:.3f}, specificity {spec:.3f}\n")
print("  patients who appear      readmitted   precision, if sensitivity and specificity held")
for k, rate in base_rate.items():
    precision = sens * rate / (sens * rate + (1 - spec) * (1 - rate))
    label = "5 or more times" if k == 5 else f"{k} time{'s' if k > 1 else ''}"
    print(f"  {label:<24} {rate:>9.1%}   {precision:>9.1%}")

# Precision: **11%** to **52%**

## Same sensitivity, same specificity.

### ⚠ *If* they held in every group. Studio question 2 checks.

# From the lab — one model, three populations

The breast ultrasound model at the threshold that catches 90% of cancers (specificity 72%).
Hold both fixed and change only who walks in:

| population | share with cancer | precision (PPV) | one cancer per … women flagged |
|---|---|---|---|
| screening everyone | about 0.5% | 2% | 62 |
| diagnostic ultrasound — this test set | 8% | 22% | 4.5 |
| BI-RADS 4 referrals | about 30% | 58% | 1.7 |

The same flag means a 2% chance of cancer or a 58% chance, depending on the population. (The
middle row is measured; the other two prevalences are illustrative round figures.)

# The unit you predict for

## Meeting 6's exit ticket: a dataset where shuffling rows would be dishonest.
## Three of you named *the same unit on both sides of the split* — patient visits,
## loan rows, a fish's movements.

## Measured.

In [ ]:
X_all, y_all, patient = d[NUM + CAT], d["readmit30"], d["patient_nbr"]

shuffled = cross_val_score(pipeline(), X_all, y_all, scoring="roc_auc",
                           cv=KFold(5, shuffle=True, random_state=0))
grouped = cross_val_score(pipeline(), X_all, y_all, scoring="roc_auc",
                          cv=GroupKFold(5), groups=patient)

print(f"  shuffled folds (a patient can be on both sides)  AUC {shuffled.mean():.4f}   "
      f"fold-to-fold SD {shuffled.std(ddof=1):.4f}")
print(f"  grouped folds  (each patient on one side)        AUC {grouped.mean():.4f}   "
      f"fold-to-fold SD {grouped.std(ddof=1):.4f}")
print(f"  difference                                           {shuffled.mean() - grouped.mean():+.4f}")

# Nothing we can see.

## +0.0004 of AUC — folds move by 0.003 on their own.

### Checked, and absent. But look at **who** is in a shuffled test set.

In [ ]:
rand_train, rand_test = train_test_split(d, test_size=0.25, random_state=764)
returning = rand_test["patient_nbr"].isin(set(rand_train["patient_nbr"]))

print(f"  a shuffled split: {returning.mean():.1%} of test encounters come from patients "
      f"who are also in training")
print(f"  readmitted within 30 days:   returning patients {rand_test.loc[returning, 'readmit30'].mean():.1%}"
      f"    new patients {rand_test.loc[~returning, 'readmit30'].mean():.1%}")

# Why: is a readmission usually itself another row? Does the patient have a LATER encounter?
later = d["encounter_id"] < d.groupby("patient_nbr")["encounter_id"].transform("max")
print(f"\n  patient has a later encounter in the data:   readmitted within 30 days "
      f"{later[d['readmit30'] == 1].mean():.1%}    not readmitted at all "
      f"{later[d['readmitted'] == 'NO'].mean():.1%}")

# Decide the unit you predict for — first

## Returning patients: **19.7%** readmitted. New patients: **5.6%**.

### A patient's first visit and "anyone who walks in" are different questions.

# From the lab — what counts as one prediction?

In the breast ultrasound data the unit question has two candidate answers: an **image**, or
an **exam** — every image of one breast from one visit. We predict once per exam — a bag of
about five images — and split by **patient**: 70% of patients to training, 20% to validation,
10% to test, so every exam from one patient sits on one side (Hansen et al., *Journal of
Imaging Informatics in Medicine*, 2026).

Split by image instead and the leak is not subtle. **29% of the 756,315 images are
near-duplicate pairs** — the same view saved twice, with and without the sonographer's
measurement marks. Put one twin in training and the other in the test set, and the "unseen"
test image is one the model has already seen.

# Studio

## `course/Day09_100626/` — your own seed, patient-grouped split

### 1. AUC, the confusion matrix at 0.5, and at a threshold **you** choose — and why.
### 2. Prior inpatient stays, none vs. 1+: does precision follow the base rate?
###    **Write down what Section 6 predicts first.**
### 3. One row per patient. Prevalence, AUC, precision — and which question is the hospital's?

## Until the exit ticket at 3:35. Done early? **Lab 5 Part 1** — same data, same split.
## Not done? Finish at home: the upload is due 3:40 tomorrow.

# Halfway

## A threshold — and a reason for it?

## Q2 prediction written down?

### Write both in your notebook — Thursday's class opens with them.

# Exit ticket

## A discharge nurse can call twenty patients a day.

## In one sentence: what does AUC 0.655 mean for that list —
## and what does the nurse need that AUC cannot say?

# Before Thursday

## **Lab 4** due tonight · **Lab 5** out today

## Thursday: **the threshold laboratory** — same model, three decisions.

## No reading. Bring today's model.

# More about today's data

- **The dataset** — [Diabetes 130-US Hospitals for Years 1999–2008, on the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008):
  what is in it, and every variable described.
- **The paper it comes from** — Strack et al. (2014), [*Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records*](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/),
  *BioMed Research International* — free on PubMed Central. How the 101,766 encounters were
  chosen from 130 US hospitals, and the clinical question the authors asked.
- **Why hospitals count readmissions at all** — [Medicare's Hospital Readmissions Reduction Program](https://www.cms.gov/medicare/payment/prospective-payment-systems/acute-inpatient-pps/hospital-readmissions-reduction-program-hrrp),
  which since 2012 has cut payments to hospitals with too many readmissions for certain
  conditions. Diabetes is not one of them.